# Module 9 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

In [1262]:
from copy import deepcopy
import random
import math

## Naive Bayes Classifier

For this assignment you will be implementing and evaluating a Naive Bayes Classifier with the same data from last week:

http://archive.ics.uci.edu/ml/datasets/Mushroom

(You should have downloaded it).

<div style="background: lemonchiffon; margin:20px; padding: 20px;">
    <strong>Important</strong>
    <p>
        No Pandas. The only acceptable libraries in this class are those contained in the `environment.yml`. No OOP, either. You can use Dicts, NamedTuples, Data Classes, etc. as your abstract data type (ADT).
    </p>
</div>


You'll first need to calculate all of the necessary probabilities using a `train` function. A flag will control whether or not you use "+1 Smoothing" or not. You'll then need to have a `classify` function that takes your probabilities, a List of instances (possibly a list of 1) and returns a List of Tuples. Each Tuple has the best class in the first position and a dict with a key for every possible class label and the associated *normalized* probability. For example, if we have given the `classify` function a list of 2 observations, we would get the following back:

```
[("e", {"e": 0.98, "p": 0.02}), ("p", {"e": 0.34, "p": 0.66})]
```

when calculating the error rate of your classifier, you should pick the class label with the highest probability; you can write a simple function that takes the Dict and returns that class label.

As a reminder, the Naive Bayes Classifier generates the *unnormalized* probabilities from the numerator of Bayes Rule:

$$P(C|A) \propto P(A|C)P(C)$$

where C is the class and A are the attributes (data). Since the normalizer of Bayes Rule is the *sum* of all possible numerators and you have to calculate them all, the normalizer is just the sum of the probabilities.

You will have the same basic functions as the last module's assignment and some of them can be reused or at least repurposed.

`train` takes training_data and returns a Naive Bayes Classifier (NBC) as a data structure. There are many options including namedtuples and just plain old nested dictionaries. **No OOP**.

```
def train(training_data, smoothing=True):
   # returns the "classifier" (however you decided to represent the probability tables).
```

The `smoothing` value defaults to True. You should handle both cases.

`classify` takes a NBC produced from the function above and applies it to labeled data (like the test set) or unlabeled data (like some new data). (This is not the same `classify` as the pseudocode which classifies only one instance at a time; it can call it though).

```
def classify(nbc, observations, labeled=True):
    # returns a list of tuples, the argmax and the raw data as per the pseudocode.
```

`evaluate` takes a data set with labels (like the training set or test set) and the classification result and calculates the classification error rate:

$$error\_rate=\frac{errors}{n}$$

Do not use anything else as evaluation metric or the submission will be deemed incomplete, ie, an "F". (Hint: accuracy rate is not the error rate!).

`cross_validate` takes the data and uses 10 fold cross validation (from Module 3!) to `train`, `classify`, and `evaluate`. **Remember to shuffle your data before you create your folds**. I leave the exact signature of `cross_validate` to you but you should write it so that you can use it with *any* `classify` function of the same form (using higher order functions and partial application). If you did so last time, you can reuse it for this assignment.

Following Module 3's material (course notes), `cross_validate` should print out a table in exactly the same format. What you are looking for here is a consistent evaluation metric cross the folds. Print the error rate to 4 decimal places. **Do not convert to a percentage.**


To summarize...

Apply the Naive Bayes Classifier algorithm to the Mushroom data set using 10 fold cross validation and the error rate as the evaluation metric. You will do this *twice*. Once with smoothing=True and once with smoothing=False. You should follow up with a brief hypothesis/explanation for the similarities or differences in the results. You may also compare the results to the Decision Tree and why you think they're different (if they are).

### Provided Functions

You do not need to document these.

You can use this function to read the data file.

In [1263]:
def parse_data(file_name: str) -> list[list]:
    data = []
    file = open(file_name, "r")
    for line in file:
        datum = line.rstrip().split(",")
        data.append(datum)
    random.shuffle(data)
    return data

You can use this function to create 10 folds for 5x2 cross validation.

In [1264]:
def create_folds(xs: list, n: int) -> list[list[list]]:
    k, m = divmod(len(xs), n)
    # be careful of generators...
    return list(xs[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n))

Put your code after this line:

-----

In [1265]:
ATTRIBUTES = [
    "eat",
    "cap-shape",
    "cap-surface",
    "cap-color",
    "bruises",
    "odor",
    "gill-attachment",
    "gill-spacing",
    "gill-size",
    "gill-color",
    "stalk-shape",
    "stalk-root",
    "stalk-surface-above-ring",
    "stalk-surface-below-ring",
    "stalk-color-above-ring",
    "stalk-color-below-ring",
    "veil-type",
    "veil-color",
    "ring-number",
    "ring-type",
    "spore-print-color",
    "population",
    "habitat"
]

<a id="clean_data"></a>
## clean_data

`clean_data` removes observations from the dataset that contain missing observations. The symbol for a missing observation defaults to "?".
* **data** list[list[str]]: the input data set. Assumed to be a list of lists of strings.
* **missing_str** str: the symbol indicating missing data. Lists containing the symbol will be removed. Defaults to "?".

**returns** list[list[str]]: the cleaned dataset.

In [1266]:
def clean_data(data: list[list[str]], missing_str: str = "?") -> list[list[str]]:
    clean = []
    for row in data:
        if missing_str in row:
            continue
        else:
            clean.append(row)
    print(f"{len(data) - len(clean)} rows removed due to missing values. "
          f"Original dataset contained {len(data)} rows. "
          f"Cleaned dataset contains {len(clean)} rows.")
    return clean
    

In [1267]:
data = [["A", "B", "C"]]
assert clean_data(data) == [["A", "B", "C"]]
data = [["A", "B", "C"], ["?"]]
assert clean_data(data) == [["A", "B", "C"]]
data = [["?"]]
assert clean_data(data) == []

0 rows removed due to missing values. Original dataset contained 1 rows. Cleaned dataset contains 1 rows.
1 rows removed due to missing values. Original dataset contained 2 rows. Cleaned dataset contains 1 rows.
1 rows removed due to missing values. Original dataset contained 1 rows. Cleaned dataset contains 0 rows.


<a id="probability_of"></a>
## probability_of

`probability_of` calculates the probability for the label that's input into the function, given the trained Naive Bayes Classifier and observation instance. **Used By**: [classify](#classify)

* **nbc** dict: the trained Naive Bayes Classifier, with prior and conditional probabilities, in a nested dict structure.
* **instance** dict: the specific observation to use for calculating the probability for the given label.
* **label** str:  the class label to evaluate.

**returns** float: returns the probability of the class label given the evidence and NBC nested dict.

In [1268]:
def probability_of(nbc: dict, instance: dict, label: str) -> float:
    prob = nbc["priors"][label]
    for k, v in instance.items():
        prob *= nbc["conditionals"][label][k][v]
    return prob
        

In [1269]:
mock_nbc = {'priors':{'e': 0.6666666666666666, 'p': 0.3333333333333333}, 
            'conditionals': {'e':
                             {'cap-shape': {'b': 0.4, 'x': 0.2, 'f': 0.4}, 'cap-surface': {'s': 0.4, 'y': 0.4, 'f': 0.2}, 'cap-color': {'e': 0.5, 'y': 0.5}},
                             'p':
                             {'cap-shape': {'b': 0.25, 'x': 0.5, 'f': 0.25}, 'cap-surface': {'s': 0.25, 'y': 0.25, 'f': 0.5}, 'cap-color': {'e': 0.3333333333333333, 'y': 0.6666666666666666}}
                              }
            }
mock_observation = {'cap-shape': 'b', 'cap-surface': 's', 'cap-color': 'y'}
e_prob = probability_of(mock_nbc,mock_observation,label="e")
p_prob = probability_of(mock_nbc,mock_observation,label="p")
expected_e_prob = 0.66666666666*0.4*0.4*0.5
expected_p_prob = 0.33333333333*0.25*0.25*0.666666666
assert e_prob > 0, p_prob > 0
assert math.isclose(e_prob, expected_e_prob, rel_tol=1e-6)
assert math.isclose(p_prob, expected_p_prob, rel_tol=1e-6)

<a id="normalize"></a>
## normalize

`normalize` takes the raw probabilities for each class label and normalizes them so that they sum to 1. It also sorts the class probabilities in descending order. **Used By**: [classify](#classify)

* **results** results: the raw probabilities from an inference of the NBC.

**returns** dict: returns the same results dict but with the values normalized and sorted by descending probability.

In [1270]:
def normalize(results: dict) -> dict:
    total = sum(results.values())
    for k, v in results.items():
        results[k] = v/total
    sorted_results = dict(sorted(results.items(), key=lambda item: item[1], reverse=True))
    return sorted_results

In [1271]:
mock_results = {"e": 0.5, "p": 0.5}
assert list(normalize(mock_results).items()) == [("e", 0.5), ("p", 0.5)]
mock_results = {"e": 0.1, "p": 0.4}
assert list(normalize(mock_results).items()) == [("p", 0.8), ("e", 0.2)]
mock_results = {"e": 0.05, "p": 0.4, "x": 0.05}
assert list(normalize(mock_results).items()) == [("p", 0.8), ("e", 0.1), ("x", 0.1)]

<a id="find_best"></a>
## find_best

`find_best` finds the class label with the highest priority and returns the label string, and higher up the call stack returns the string from the NBC train function. **Used By**: [train](#train)

* **results** dict: a dict containing each class label and its probability.

**returns** str: returns the class label with highest probability.

In [1272]:
def find_best(results: dict) -> str:
    best = max(results, key=results.get)
    return best

In [1273]:
mock_results = {"e": 0.5, "p": 0.5}
assert find_best(mock_results) == "e"
mock_results = {"p": 0.8, "e": 0.2}
assert find_best(mock_results) == "p"
mock_results = {"p": 0.8, "e": 0.1, "x": 0.1}
assert find_best(mock_results) == "p"

<a id="get_attr_types"></a>
## get_attr_types

`get_attr_types` creates a dictionary of sets that contain all the known values for each feature based on the training data. This is required to correctly count feature value combinations for training the NBC. **Used By**: [train](#train)

* **training_data** list[dict]: The training data as a list of dicts.
* **attributes** set[str]: The feature attributes from which to compile the possible values for each feature.

**returns** dict[str, set[str]]: returns a dictionary that contains all the known possible values that each feature can take on.

In [1274]:
def get_attr_types(training_data: list[dict], attributes: set[str]) -> dict[str, set[str]]:   
    attr_types_dict = {attr: set() for attr in attributes}
    for row in training_data:
        for attr in attributes:
            attr_types_dict[attr].add(row[attr])
    return attr_types_dict

In [1275]:
mock_training_data = [{'eat': 'e', 'cap-shape': 'b', 'cap-surface': 's', 'cap-color': 'y', 'bruises': 't', 'odor': 'l', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'k', 'stalk-shape': 'e', 'stalk-root': 'c', 'stalk-surface-above-ring': 's', 'stalk-surface-below-ring': 's', 'stalk-color-above-ring': 'w', 'stalk-color-below-ring': 'w', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'p', 'spore-print-color': 'n', 'population': 's', 'habitat': 'm'},
                      {'eat': 'p', 'cap-shape': 'x', 'cap-surface': 'f', 'cap-color': 'y', 'bruises': 'f', 'odor': 'f', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'p', 'stalk-shape': 'e', 'stalk-root': 'b', 'stalk-surface-above-ring': 'k', 'stalk-surface-below-ring': 'k', 'stalk-color-above-ring': 'b', 'stalk-color-below-ring': 'n', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'l', 'spore-print-color': 'h', 'population': 'v', 'habitat': 'g'},
                      {'eat': 'e', 'cap-shape': 'f', 'cap-surface': 'y', 'cap-color': 'e', 'bruises': 't', 'odor': 'n', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'n', 'stalk-shape': 't', 'stalk-root': 'b', 'stalk-surface-above-ring': 's', 'stalk-surface-below-ring': 's', 'stalk-color-above-ring': 'p', 'stalk-color-below-ring': 'g', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'p', 'spore-print-color': 'n', 'population': 'v', 'habitat': 'd'}]
attr_types_dict = get_attr_types(mock_training_data, ATTRIBUTES[1:])
assert attr_types_dict["cap-shape"] == {"b", "x", "f"}
assert attr_types_dict["cap-color"] == {"e", "y"}
assert attr_types_dict["gill-spacing"] == {"c"}

<a id="get_class_totals"></a>
## get_class_totals

`get_class_totals` calculates the overall occurrences of each class label as part of the Naive Bayes Classifier function. **Used By**: [train](#train)

* **training_data** list[dict]: The training data as a list of dicts.
* **target_attr** str: The target attribute.

**returns** dict: returns a dict with the total counts for each target class.

In [1276]:
def get_class_totals(training_data: list[dict], target_attr: str) -> dict:
    class_totals_dict = {}
    for row in training_data:
        if row[target_attr] not in class_totals_dict:
            class_totals_dict[row[target_attr]] = 0
        class_totals_dict[row[target_attr]] += 1
    return class_totals_dict

In [1277]:
mock_training_data = [{'eat': 'e'}, {'eat': 'p'},{'eat': 'x', },{'eat': 'x', },{'eat': 'x', }]
class_totals_dict = get_class_totals(mock_training_data, "eat")
assert class_totals_dict["e"] == 1
assert class_totals_dict["p"] == 1
assert class_totals_dict["x"] == 3

<a id="get_counts"></a>
## get_counts

`get_counts` totals the counts for each possible feature value, over all attributes as a helper function in the main Naive Bayes Classifier training function. **Used By**: [train](#train)

* **training_data** list[dict]: The training data as a list of dicts.
* **attributes** set[str]: The feature attributes from which to total the counts.
* **target_attr** str: The target attribute.

**returns** dict: returns a dictionary containing the counts for each value that each attribute takes on.

In [1278]:
def get_counts(training_data: list[dict],  attributes: set[str], target_attr: str = "eat") -> dict:
    target_dict = {}
    for row in training_data:
        tgt_val = row[target_attr]
        if tgt_val not in target_dict:
            target_dict[tgt_val] = {}
        for attr in attributes:
            attr_val = row[attr]
            if attr not in target_dict[tgt_val]:
                target_dict[tgt_val][attr] = {}
            if attr_val not in target_dict[tgt_val][attr]:
                target_dict[tgt_val][attr][attr_val] = 0
            target_dict[tgt_val][attr][attr_val] += 1
    return target_dict

In [1279]:
mock_training_data = [{'eat': 'e', 'cap-shape': 'b', 'cap-surface': 's', 'cap-color': 'y', 'bruises': 't', 'odor': 'l', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'k', 'stalk-shape': 'e', 'stalk-root': 'c', 'stalk-surface-above-ring': 's', 'stalk-surface-below-ring': 's', 'stalk-color-above-ring': 'w', 'stalk-color-below-ring': 'w', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'p', 'spore-print-color': 'n', 'population': 's', 'habitat': 'm'},
                      {'eat': 'p', 'cap-shape': 'x', 'cap-surface': 'f', 'cap-color': 'y', 'bruises': 'f', 'odor': 'f', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'p', 'stalk-shape': 'e', 'stalk-root': 'b', 'stalk-surface-above-ring': 'k', 'stalk-surface-below-ring': 'k', 'stalk-color-above-ring': 'b', 'stalk-color-below-ring': 'n', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'l', 'spore-print-color': 'h', 'population': 'v', 'habitat': 'g'},
                      {'eat': 'e', 'cap-shape': 'f', 'cap-surface': 'y', 'cap-color': 'e', 'bruises': 't', 'odor': 'n', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'n', 'stalk-shape': 't', 'stalk-root': 'b', 'stalk-surface-above-ring': 's', 'stalk-surface-below-ring': 's', 'stalk-color-above-ring': 'p', 'stalk-color-below-ring': 'g', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'p', 'spore-print-color': 'n', 'population': 'v', 'habitat': 'd'}]
count_dict = get_counts(mock_training_data, ATTRIBUTES[1:])
assert count_dict["e"]["cap-shape"]["b"] == 1
assert count_dict["e"]["bruises"]["t"] == 2
assert count_dict["p"]["cap-color"]["y"] == 1

<a id="train"></a>
## train

`train` trains a Naive Bayesian Classifier by totaling the overall class probabilities, and then each conditioanl probability for every combination of feature value and class value. It also optionally implements +1 smooths during calculations. **Uses**: [get_counts](#get_counts), [get_attr_types](#get_attr_types), [get_class_totals](#get_class_totals) **Used By**: [cross_validate](#cross_validate)

* **training_data** list[dict]: The training data as a list of dicts.
* **attributes** set[str]: The feature attributes from which to calculate the prior and conditional probabilities.
* **target_attr** str: The target attribute.
* **smoothing** bool: A flag for whether to implement +1 smoothing when calculating the conditional probabilities.

**returns** dict: returns a dictionary with "prior" and "conditionals" keys that contain all the necessary probabilities to do inference.

In [1280]:
def train(training_data: list[dict],  attributes: set[str], target_attr: str = "eat", smoothing=True) -> dict:
   count_dict = get_counts(training_data, attributes, target_attr)
   attr_types_dict = get_attr_types(training_data, attributes)
   class_totals_dict = get_class_totals(training_data, target_attr)
   class_probs_dict = {key: class_totals_dict[key]/sum(class_totals_dict.values()) for key in class_totals_dict.keys()}
   prob_dict = {}
   for tgt_val, attr_dict in count_dict.items():
      prob_dict[tgt_val] = {}
      denominator = class_totals_dict[tgt_val]
      for attr, attr_type_dict in attr_dict.items():
         prob_dict[tgt_val][attr] = {}
         num_unique_attrs = len(attr_types_dict[attr])
         for attr_val in attr_types_dict[attr]:
            attr_count = attr_type_dict.get(attr_val, 0)
            if smoothing:
               prob_dict[tgt_val][attr][attr_val] = (attr_count + 1) / (denominator + num_unique_attrs)
            else:
               prob_dict[tgt_val][attr][attr_val] = attr_count / denominator
   return {"priors": class_probs_dict, "conditionals": prob_dict}

In [1281]:
mock_training_data = [{'eat': 'e', 'cap-shape': 'b', 'cap-surface': 's', 'cap-color': 'y', 'bruises': 't', 'odor': 'l', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'k', 'stalk-shape': 'e', 'stalk-root': 'c', 'stalk-surface-above-ring': 's', 'stalk-surface-below-ring': 's', 'stalk-color-above-ring': 'w', 'stalk-color-below-ring': 'w', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'p', 'spore-print-color': 'n', 'population': 's', 'habitat': 'm'},
                      {'eat': 'p', 'cap-shape': 'x', 'cap-surface': 'f', 'cap-color': 'y', 'bruises': 'f', 'odor': 'f', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'p', 'stalk-shape': 'e', 'stalk-root': 'b', 'stalk-surface-above-ring': 'k', 'stalk-surface-below-ring': 'k', 'stalk-color-above-ring': 'b', 'stalk-color-below-ring': 'n', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'l', 'spore-print-color': 'h', 'population': 'v', 'habitat': 'g'},
                      {'eat': 'e', 'cap-shape': 'f', 'cap-surface': 'y', 'cap-color': 'e', 'bruises': 't', 'odor': 'n', 'gill-attachment': 'f', 'gill-spacing': 'c', 'gill-size': 'b', 'gill-color': 'n', 'stalk-shape': 't', 'stalk-root': 'b', 'stalk-surface-above-ring': 's', 'stalk-surface-below-ring': 's', 'stalk-color-above-ring': 'p', 'stalk-color-below-ring': 'g', 'veil-type': 'p', 'veil-color': 'w', 'ring-number': 'o', 'ring-type': 'p', 'spore-print-color': 'n', 'population': 'v', 'habitat': 'd'}]
nbc = train(mock_training_data, ATTRIBUTES[1:], smoothing=False)
assert nbc["priors"] == {'e': 0.6666666666666666, 'p': 0.3333333333333333}
assert nbc["conditionals"]["e"]["cap-shape"] == {"b": 0.5, "x": 0.0, "f": 0.5}
nbc = train(mock_training_data, ATTRIBUTES[1:], smoothing=True)
assert nbc["conditionals"]["e"]["cap-shape"] == {"b": 0.4, "x": 0.2, "f": 0.4}

<a id="classify"></a>
## classify

`classify` calculates the probability of each class label, given a single observation, using the Naive Bayes Classify psuedocode. Part of the larger Naive Bayes Classifier pipeline. **Uses**: [probability_of](#probability_of), [normalize](#normalize), [find_best](#find_best) **Used By**: [classify_all](#classify_all)

* **nbc** dict: the trained Naive Bayes Classifier, with prior and conditional probabilities, in a nested dict structure.
* **instance** dict: the specific observation to use for calculating the probability for the given label.

**returns** tuple[str, dict]: returns a tuple of the most likely class label, and a dict of all of the class probabilities.

In [1282]:
def classify(nbc: dict, instance: dict) -> tuple[str, dict]:
    results = {}
    for label in nbc["conditionals"]:
        results[label] = probability_of(nbc, instance, label)
    results = normalize(results)
    best = find_best(results)
    return best, results
        

In [1283]:
mock_training_data = [{'eat': 'e', 'cap-shape': 'x', 'cap-surface': 's', 'cap-color': 'y'},
                      {'eat': 'p', 'cap-shape': 'x', 'cap-surface': 'f', 'cap-color': 'e'},
                      {'eat': 'e', 'cap-shape': 'f', 'cap-surface': 'f', 'cap-color': 'y'}]
mock_test_instance = {'cap-shape': 'f', 'cap-surface': 's', 'cap-color': 'e'}

nbc = train(mock_training_data, ["cap-shape", "cap-surface", "cap-color"], smoothing=True)
result = classify(nbc,mock_test_instance)
expected_nbc = {'priors': {'e': 0.6666666666666666, 'p': 0.3333333333333333},
                'conditionals': {'e': {'cap-shape': {'x': 0.5, 'f': 0.5},
                                       'cap-surface': {'s': 0.5, 'f': 0.5},
                                       'cap-color': {'e': 0.25, 'y': 0.75}},
                                'p': {'cap-shape': {'x': 0.6666666666666666, 'f': 0.3333333333333333},
                                      'cap-surface': {'s': 0.3333333333333333, 'f': 0.6666666666666666},
                                      'cap-color': {'e': 0.6666666666666666, 'y': 0.3333333333333333}}}}
expected_e_prob = 0.666666*0.5*0.5*0.25
expected_p_prob = 0.333333*0.333333*0.333333*0.666666
expected_norm_e_prob = expected_e_prob/(expected_e_prob+expected_p_prob)
expected_norm_p_prob = expected_p_prob/(expected_e_prob+expected_p_prob)
expected_result = ("e", {"e": 0.6279076, "p": 0.372092})
assert result[0] == expected_result[0]
assert math.isclose(result[1]["e"], expected_result[1]["e"], rel_tol=1e-4)
assert math.isclose(result[1]["p"], expected_result[1]["p"], rel_tol=1e-4)

<a id="classify_all"></a>
## classify_all

`classify_all` finds the most likely class label and set of probabilities for each label, for a series of observations. It is essentially a wrapper around the classify function. If an observation already has a truth label, it's removed before running the classifier. **Uses**: [classify](#classify) **Used By**: [cross_validate](#cross_validate)

* **nbc** dict: the trained Naive Bayes Classifier, with prior and conditional probabilities, in a nested dict structure.
* **observations** list[dict]: the list of observations to use for inference.
* **labeled** bool: a flag that indicates whether to expect the truth label as part of each observation or not.
* **tgt_attr** str: the target label that may be removed from the observation depending on the value of the labeled flag.

**returns** type: returns a list of inferences based on the observations, where each result is a tuple with the most likely class label and associated dict with the probability of all class labels.

In [1284]:
def classify_all(nbc: dict, observations: list[dict], labeled: bool = True, tgt_attr: str = "eat") -> list[tuple[str, dict]]:
    results = []
    for row in observations:
        instance = row.copy()
        if labeled:
            del instance[tgt_attr]
        result = classify(nbc, instance)
        results.append(result)
    return results

In [1285]:
mock_training_data = [{'eat': 'e', 'cap-shape': 'x', 'cap-surface': 's', 'cap-color': 'y'},
                      {'eat': 'p', 'cap-shape': 'x', 'cap-surface': 'f', 'cap-color': 'e'},
                      {'eat': 'e', 'cap-shape': 'f', 'cap-surface': 'f', 'cap-color': 'y'}]
mock_test_instance_labeled = [{'eat': 'e', 'cap-shape': 'f', 'cap-surface': 's', 'cap-color': 'e'}]
mock_test_instance_unlabeled = [{'cap-shape': 'f', 'cap-surface': 's', 'cap-color': 'e'}]

nbc = train(mock_training_data, ["cap-shape", "cap-surface", "cap-color"], smoothing=True)
results_labeled = classify_all(nbc=nbc,observations=mock_test_instance_labeled, labeled=True)
results_unlabeled = classify_all(nbc=nbc,observations=mock_test_instance_unlabeled, labeled=False)
expected_nbc = {'priors': {'e': 0.6666666666666666, 'p': 0.3333333333333333},
                'conditionals': {'e': {'cap-shape': {'x': 0.5, 'f': 0.5},
                                       'cap-surface': {'s': 0.5, 'f': 0.5},
                                       'cap-color': {'e': 0.25, 'y': 0.75}},
                                'p': {'cap-shape': {'x': 0.6666666666666666, 'f': 0.3333333333333333},
                                      'cap-surface': {'s': 0.3333333333333333, 'f': 0.6666666666666666},
                                      'cap-color': {'e': 0.6666666666666666, 'y': 0.3333333333333333}}}}
expected_e_prob = 0.666666*0.5*0.5*0.25
expected_p_prob = 0.333333*0.333333*0.333333*0.666666
expected_norm_e_prob = expected_e_prob/(expected_e_prob+expected_p_prob)
expected_norm_p_prob = expected_p_prob/(expected_e_prob+expected_p_prob)
expected_result = ("e", {"e": 0.6279076, "p": 0.372092})

assert results_labeled == results_unlabeled
assert len(results_labeled) == 1
assert result[0] == expected_result[0]
assert math.isclose(result[1]["e"], expected_result[1]["e"], rel_tol=1e-4)
assert math.isclose(result[1]["p"], expected_result[1]["p"], rel_tol=1e-4)

<a id="evaluate"></a>
## evaluate

`evaluate` determines the error rate of a dataset, by comparing the predicted labels to the true labels. The ordered list of observations is assumed to correspond to the ordered list of inferences used as inputs.

* **observations** list[dict]: the dataset for evaluation.
* **inferences** list[tuple[str, dict]]: the Naive Bayes Classifier inferences, provided earlier up the call stack by the classify_all function.
* **tgt_attr** str: the target attribute name to use for the truth label.

**returns** type: returns the error rate of the given dataset.

In [1286]:
def evaluate(observations: list[dict], inferences: list[tuple[str, dict]], tgt_attr: str = "eat") -> float:
    n = len(observations)
    errors = 0
    for obs, inference in zip(observations, inferences):
        if obs[tgt_attr] != inference[0]:
            errors += 1
    error_rate = errors / n
    return error_rate

In [1287]:
observations = [{"eat": "p", "foo": "bar"}]
inferences = [("p", {})]
assert evaluate(observations, inferences) == 0.0
observations = [{"eat": "p", "foo": "bar"}]
inferences = [("e", {})]
assert evaluate(observations, inferences) == 1.0
observations = [{"eat": "p", "foo": "bar"}, {"eat": "p"}]
inferences = [("p", {}), ("e", {})]
assert evaluate(observations, inferences) == 0.5

<a id="cross_validate"></a>
## cross_validate

`cross_validate` runs cross validation using the observations, specified set of attributes and the number of folds. It trains, and then classifies and evaluates on both the training set and test set. The error rate is then printed out for both and added to lists that are returned for debugging purposes. **Uses**: [classify_all](#classify_all), [evaluate](#evaluate), [create_folds](#create_folds), [train](#train)

* **observations** list[dict]: the input dataset for validation that will create folds and then train and evaluate for each split.
* **attributes** set[str]: the attribute set containing all the features of the dataset.
* **num_folds** int: the number of splits for cross validation in the dataset.
* **smoothing** bool: whether to use +1 smoothing or not.

**returns** tuple[list, list]: returns the list of training and test errors.

In [1288]:
def cross_validate(observations: list[dict], attributes: set[str], num_folds: int, smoothing: bool) -> tuple[list, list]:
    random.seed(42)
    random.shuffle(observations)
    folded_data = create_folds(observations, num_folds)
    train_error = []
    test_error = []
    for i in range(len(folded_data)):
        train_folds = folded_data[:i] + folded_data[i+1:]
        training_set = sum(train_folds, [])
        test_set = folded_data[i]
        nbc = train(training_data=training_set, attributes=attributes, smoothing=smoothing)
        train_classified = classify_all(nbc, deepcopy(training_set))
        test_classified = classify_all(nbc, deepcopy(test_set))
        train_error_rate = evaluate(training_set, train_classified)
        test_error_rate = evaluate(test_set, test_classified)
        print(f"Fold {i + 1} error rates, train: {train_error_rate:.4f} test: {test_error_rate:.4f}")
        train_error.append(train_error_rate)
        test_error.append(test_error_rate)
    return train_error, test_error

In [1289]:
data = [{"eat": "e", "color": "red"},{"eat": "e", "color": "red"},{"eat": "p", "color": "green"},{"eat": "p", "color": "green"}]
train_err, test_err = cross_validate(data, {"color"}, 2, smoothing=True)
assert train_err == [0.0, 0.0]
assert test_err == [0.0, 0.0]
data = [{"eat": "p", "color": "green"},{"eat": "p", "color": "yellow"},{"eat": "e", "color": "green"},{"eat": "e", "color": "yellow"}]
train_err, test_err = cross_validate(data, {"color"}, 2, smoothing=True)
print(train_err, test_err)
assert train_err == [0.0, 0.0]
assert test_err == [1.0, 1.0]

Fold 1 error rates, train: 0.0000 test: 0.0000
Fold 2 error rates, train: 0.0000 test: 0.0000
Fold 1 error rates, train: 0.0000 test: 1.0000
Fold 2 error rates, train: 0.0000 test: 1.0000
[0.0, 0.0] [1.0, 1.0]


This is the "pipeline" that runs all the functions discussed above. It prints the train and test error rates for each fold.

In [1290]:
data = parse_data(file_name="agaricus-lepiota.data") # this file should be in the same directory as the script
cleaned_data = clean_data(data)
formatted_data = [dict(zip(ATTRIBUTES, row)) for row in cleaned_data]
root_attributes = ATTRIBUTES[1:]
print("CROSS VALIDATION SMOOTHING ON")
train_err, test_err = cross_validate(formatted_data, set(root_attributes), 10, smoothing=True)
print("CROSS VALIDATION SMOOTHING OFF")
train_err, test_err = cross_validate(formatted_data, set(root_attributes), 10, smoothing=False)


2480 rows removed due to missing values. Original dataset contained 8124 rows. Cleaned dataset contains 5644 rows.
CROSS VALIDATION SMOOTHING ON
Fold 1 error rates, train: 0.0248 test: 0.0372
Fold 2 error rates, train: 0.0272 test: 0.0354
Fold 3 error rates, train: 0.0252 test: 0.0212
Fold 4 error rates, train: 0.0311 test: 0.0442
Fold 5 error rates, train: 0.0217 test: 0.0248
Fold 6 error rates, train: 0.0250 test: 0.0160
Fold 7 error rates, train: 0.0240 test: 0.0213
Fold 8 error rates, train: 0.0207 test: 0.0124
Fold 9 error rates, train: 0.0281 test: 0.0160
Fold 10 error rates, train: 0.0224 test: 0.0319
CROSS VALIDATION SMOOTHING OFF
Fold 1 error rates, train: 0.0028 test: 0.0000
Fold 2 error rates, train: 0.0028 test: 0.0018
Fold 3 error rates, train: 0.0030 test: 0.0018
Fold 4 error rates, train: 0.0026 test: 0.0053
Fold 5 error rates, train: 0.0026 test: 0.0035
Fold 6 error rates, train: 0.0026 test: 0.0053
Fold 7 error rates, train: 0.0028 test: 0.0018
Fold 8 error rates, trai

From these results, it seems that the error rate when smoothing is off is less than when smoothing is on. This could have something to do with the determinism in the dataset that we saw particularly with the decision tree. That is, this dataset isn't from a stochastic source, it's from a reference manual. The Naive Bayes classifier performs quite well on the dataset, but at the same time doesn't appear to be just memorizing the rule structure as was the case with my decision tree.

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.